# Movement feature extraction from preprocessed signals

This notebook extracts three movement features (`QoM`, `amplitude_sum`, `mean_speed`) from the preprocessed signals produced by the preprocessing pipeline, and writes a final participant-level CSV ready for statistical analysis.

**Design principles**

- All computation lives in `sources/movement_analysis.py`; this notebook only orchestrates the pipeline and performs optional visual checks.
- One cell = one conceptual step.
- Every modifiable parameter (paths, sampling rate, peak-detection thresholds, inspection target) is grouped in a single cell near the top.
- Visual-check blocks are commented out by default so the notebook stays silent when executed from `main.ipynb`. Uncomment any cell locally to inspect the intermediate results.
- No preprocessing is redone here: the input CSV is assumed to contain the final, filtered, centered signals.

## 1. Imports and local module

We import the scientific Python stack and the project-local module `movement_analysis` from `sources/`. The `sources/` folder is added to `sys.path` at runtime.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ne pas redefinir PROJECT_DIR s'il existe déjà (passé par main.ipynb)
if 'PROJECT_DIR' not in locals():
    PROJECT_DIR = Path.cwd()
    if PROJECT_DIR.name == "notebooks":
        PROJECT_DIR = PROJECT_DIR.parent

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
SOURCES_DIR = PROJECT_DIR / "sources"
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"

if str(SOURCES_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCES_DIR))

import movement_analysis as ma

# Reload `movement_analysis` automatically whenever it changes on disk.
%load_ext autoreload
%autoreload 2

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("Module `movement_analysis` imported successfully.")

Module `movement_analysis` imported successfully.


## 2. Pipeline parameters

All editable parameters are gathered here:

- input and output paths,
- sampling rate (`FPS`), used for duration normalization of `QoM` and `amplitude_sum`,
- peak-detection parameters forwarded to `scipy.signal.find_peaks` (see Section 3 for their meaning),
- the participant and signal targeted by the optional visual checks.

To explore different peak-detection thresholds, edit this cell and rerun the notebook.

Peak distance and peak prominence were chosen ....

In [3]:
# --- Paths ---
INPUT_PATH = DATA_DIR / "all_participants_preprocessed_movements.csv"
OUTPUT_PATH = DATA_DIR / "all_participants_movement_features.csv"

# --- Sampling rate (must match the rate used in preprocessing) ---
FPS = 30.0  # frames per second

# --- Peak-detection parameters (find_peaks) ---
PEAK_DISTANCE = 5        # minimum separation between two peaks, in frames
PEAK_PROMINENCE = 0.025  # minimum vertical prominence of a peak
PEAK_WIDTH = None        # minimum peak width in frames (None = no constraint)
PEAK_HEIGHT = None       # minimum absolute peak height (None = no constraint)

# --- Inspection target (one participant, one signal) ---
#TEST_PID = "SZ10"
#TEST_SIGNAL = "swaying"

## 3. Feature definitions

**`find_peaks`** detects local extrema in a one-dimensional signal. Maxima are found by running the algorithm on the raw signal; minima are found by running it on the negated signal. The parameters `distance`, `prominence`, `width` and `height` filter out peaks that are too close to each other, too flat, too narrow, or too low.

**Amplitude.** All extrema (maxima and minima) are ordered chronologically, their signal values are read, and the absolute differences between successive values are summed. The result captures the cumulative oscillatory excursion of the signal. This sum is then divided by the signal duration (in seconds) to obtain `amplitude_sum`.

**QoM (Quantity of Motion).** Total number of extrema (maxima + minima) in the signal, divided by the signal duration in seconds. Higher QoM = more frequent direction reversals.

**`mean_speed`.** Mean of the absolute frame-to-frame differences of the signal. Because it is already a time average, it is **not** further normalized by duration.

For the two aggregated hand movement types (`hand_horizontal`, `hand_vertical`), features are computed independently on the left and right signals and then averaged.

## 4. Load the preprocessed CSV

We load the global preprocessed file and keep a reference to it in `df`. Inspection prints are disabled by default.

In [4]:
df = pd.read_csv(INPUT_PATH)

# --- Optional inspection (uncomment to display) ---
# print(f"Shape              : {df.shape}")
# print(f"Unique participants: {df['participant_id'].nunique()}")
# print(f"Groups             : {sorted(df['group'].unique())}")
# print(f"Columns            : {list(df.columns)}")
# df.head()

## 5. Signal lists and aggregation rules

The module exposes three module-level constants: `BASE_SIGNALS` (the eight individual signals), `MOVEMENT_TYPES` (the six aggregated movement types reported in the output), and `HAND_AGGREGATION` (the mapping used to average left and right hand signals).

In [5]:
# --- Optional inspection (uncomment to display) ---
print(f"Base signals    ({len(ma.BASE_SIGNALS)}): {ma.BASE_SIGNALS}")
print(f"Movement types  ({len(ma.MOVEMENT_TYPES)}): {ma.MOVEMENT_TYPES}")
print(f"Hand aggregation: {ma.HAND_AGGREGATION}")

Base signals    (8): ['left_hand_horizontal', 'right_hand_horizontal', 'left_hand_vertical', 'right_hand_vertical', 'swaying', 'sideways', 'head_horizontal', 'head_vertical']
Movement types  (6): ['hand_horizontal', 'hand_vertical', 'swaying', 'sideways', 'head_horizontal', 'head_vertical']
Hand aggregation: {'hand_horizontal': ['left_hand_horizontal', 'right_hand_horizontal'], 'hand_vertical': ['left_hand_vertical', 'right_hand_vertical']}


In [6]:
#signal_test = df.loc[df["participant_id"] == TEST_PID, TEST_SIGNAL].to_numpy()

#idx_max_test, idx_min_test = ma.detect_extrema(
#    signal_test,
#    distance=PEAK_DISTANCE, prominence=PEAK_PROMINENCE,
#    width=PEAK_WIDTH, height=PEAK_HEIGHT,
#)

# --- Optional inspection (uncomment to display) ---
# print(f"Participant {TEST_PID}, signal {TEST_SIGNAL}:")
# print(f"  n_frames = {len(signal_test)}")
# print(f"  n_maxima = {len(idx_max_test)}")
# print(f"  n_minima = {len(idx_min_test)}")

## 7. Per-signal features on the inspection target

We call `compute_signal_metrics` on the same (participant, signal) pair. The returned values are already duration-normalized for `QoM` and `amplitude_sum`.

In [7]:
#metrics_test = ma.compute_signal_metrics(
#    signal_test, fps=FPS,
#    distance=PEAK_DISTANCE, prominence=PEAK_PROMINENCE,
#    width=PEAK_WIDTH, height=PEAK_HEIGHT,
#)

# --- Optional inspection (uncomment to display) ---
# print(f"Metrics for {TEST_PID} / {TEST_SIGNAL} (duration-normalized):")
# for k, v in metrics_test.items():
#     print(f"  {k:14s} = {v}")

## 8. Visual check: signal with detected extrema

Plots the inspection signal with maxima (red) and minima (blue) overlaid. Useful for verifying that `PEAK_DISTANCE` and `PEAK_PROMINENCE` are neither too permissive nor too strict.

In [8]:
# --- Optional visualization (uncomment to display) ---
# sub = df[df["participant_id"] == TEST_PID].sort_values("frame").reset_index(drop=True)
# frames_test = sub["frame"].to_numpy()
# signal_test = sub[TEST_SIGNAL].to_numpy()
#
# idx_max_test, idx_min_test = ma.detect_extrema(
#     signal_test,
#     distance=PEAK_DISTANCE, prominence=PEAK_PROMINENCE,
#     width=PEAK_WIDTH, height=PEAK_HEIGHT,
# )
#
# plt.figure(figsize=(11, 4))
# plt.plot(frames_test, signal_test, color="black", linewidth=1, label="signal")
# plt.plot(frames_test[idx_max_test], signal_test[idx_max_test], "^",
#          color="tab:red", markersize=7, label=f"maxima (n={len(idx_max_test)})")
# plt.plot(frames_test[idx_min_test], signal_test[idx_min_test], "v",
#          color="tab:blue", markersize=7, label=f"minima (n={len(idx_min_test)})")
# plt.axhline(0, color="grey", linewidth=0.5)
# plt.title(f"Participant {TEST_PID} -- signal: {TEST_SIGNAL}")
# plt.xlabel("frame")
# plt.ylabel("value (shoulder-width units)")
# plt.legend()
# plt.tight_layout()
# plt.show()

## 9. Visual check: amplitude distribution

Histogram of the absolute differences between successive extrema for the inspection signal. Useful for spotting signals dominated by small oscillations or by a few large excursions.

In [9]:
# --- Optional visualization (uncomment to display) ---
# all_idx_test = np.sort(np.concatenate([idx_max_test, idx_min_test]))
# if all_idx_test.size >= 2:
#     amplitudes_test = np.abs(np.diff(signal_test[all_idx_test]))
# else:
#     amplitudes_test = np.array([])
#
# plt.figure(figsize=(8, 4))
# if amplitudes_test.size > 0:
#     plt.hist(amplitudes_test, bins=20, color="tab:purple", edgecolor="white")
#     plt.axvline(amplitudes_test.mean(), color="black", linestyle="--",
#                 label=f"mean = {amplitudes_test.mean():.4f}")
#     plt.legend()
# else:
#     plt.text(0.5, 0.5, "No amplitudes available (fewer than 2 extrema)",
#              ha="center", va="center", transform=plt.gca().transAxes)
# plt.title(f"Amplitudes between successive extrema -- {TEST_PID} / {TEST_SIGNAL}")
# plt.xlabel("amplitude (shoulder-width units)")
# plt.ylabel("count")
# plt.tight_layout()
# plt.show()

## 10. Visual check: metric summary

One-row summary of the metrics computed for the inspection pair.

In [10]:
# --- Optional inspection (uncomment to display) ---
# inspection_row = {
#     "participant_id": TEST_PID,
#     "signal": TEST_SIGNAL,
#     **metrics_test,
# }
# pd.DataFrame([inspection_row])

## 11. Per-signal features for all participants

Loop over every participant and every base signal. Missing or all-NaN signals are silently skipped (see the module docstring for robustness details).

In [11]:
per_signal_df = ma.compute_per_signal_features(
    df, fps=FPS,
    distance=PEAK_DISTANCE, prominence=PEAK_PROMINENCE,
    width=PEAK_WIDTH, height=PEAK_HEIGHT,
)

# --- Optional inspection (uncomment to display) ---
# print(f"Per-signal results: {per_signal_df.shape}")
# per_signal_df.head(10)

## 12. Aggregate left/right hand signals

The two hand movement types (`hand_horizontal` and `hand_vertical`) are built by averaging the features of the corresponding left and right base signals. The trunk and head movement types are carried through unchanged.

In [12]:
features_df = ma.aggregate_hand_signals(per_signal_df)

# --- Optional inspection (uncomment to display) ---
# print(f"Final features shape : {features_df.shape}")
#print(f"Unique participants  : {features_df['participant_id'].nunique()}")
#print(f"Movement types       : {sorted(features_df['movement_type'].unique())}")
# features_df.head(12)

## 13. Visual check: features for the inspection participant

Six-row view of the final features for `TEST_PID`, one row per movement type.

In [13]:
# --- Optional inspection (uncomment to display) ---
# features_df[features_df["participant_id"] == TEST_PID]

## 14. Save the final features CSV

Write the participant-level features file to `data/all_participants_movement_features.csv`. This file is the input of downstream statistical analyses.

In [14]:
output_path = ma.save_features(features_df, DATA_DIR)

# --- Optional inspection (uncomment to display) ---
print(f"Saved movement features -> {output_path}")
# print(f"Shape: {features_df.shape}")

Saved movement features -> c:\Users\factory\Desktop\pons.lea\data\all_participants_movement_features.csv


In [ ]:
#-- Optional summary of number of participants and valid feature values per movement type (uncomment to display) ---
# features_summary = features_df.groupby("movement_type").apply(
#    lambda df: pd.Series({
#        "n_participants_total": df["participant_id"].nunique(),
#        "n_QoM_ok": df["QoM"].notna().sum(),
#        "n_amplitude_sum_ok": df["amplitude_sum"].notna().sum(),
#        "n_mean_speed_ok": df["mean_speed"].notna().sum(),
#    })
#).reset_index()

#features_summary

## 15. Reload and summary by group

Reread the file as downstream notebooks will, and display the group-level means as a final sanity check.

In [16]:
# --- Optional inspection (uncomment to display) ---
# check = pd.read_csv(output_path)
# print(f"Reloaded shape       : {check.shape}")
# print(f"Unique participants  : {check['participant_id'].nunique()}")
# print(f"Movement types       : {sorted(check['movement_type'].unique())}")
# print(f"Columns              : {list(check.columns)}")
# print()
# print("Mean of each feature per (group, movement_type):")
# check.groupby(["group", "movement_type"])[ma.FEATURE_COLS].mean().round(4)